# সহজ ভাষায় Notebook Guide

Class 1-এর RAG API চলে। কিন্তু চলার সময় কী হচ্ছে — কত দ্রুত, কত token, উত্তর ঠিক কিনা, কখন মানুষকে ডাকতে হবে — সেটা দেখার ব্যবস্থাই **monitoring**। এই notebook project-এর একই `app/telemetry/` code চালিয়ে প্রতিটা অংশ খুলে দেখায়। Technical term English-এ, explanation Bangla-তে।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে run করুন। Kernel = project-এর `venv`।
2. Section 1-3, 5-6 real Ollama (`llama3.1:8b`) লাগে। বাকিগুলো Ollama ছাড়াও চলে।
3. Phoenix, Prometheus, Docker — কিছুই লাগে না। Trace এখানে in-memory exporter-এ ধরা হয়।

# LLM Observability & Monitoring — Exploration

1. Observer seam: traffic পাঠিয়ে request log দেখা
2. Latency কোথায় যায় — stage breakdown, prefill-এর দুই চূড়া
3. Prometheus metrics — bucket, cardinality, আর 10 s-এর ফাঁদ
4. Cheap quality signals — invented number আর ভুল citation ধরা
5. Not-found review — guardrail-এর false negative খোঁজা
6. Offline LLM judge — তিনটা design, মাপা ফলাফল
7. Traces — span tree আর GenAI attributes
8. SLO + burn rate — কখন page করবেন

In [1]:
import sys, os, tempfile, time, json, asyncio
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

tmp = Path(tempfile.mkdtemp(prefix="rag_mon_nb_"))           # notebook-only index + log
os.environ["RAG_CHROMA_DIR"] = str(tmp / "chroma")
os.environ["RAG_MONITOR_DB"] = str(tmp / "monitor.db")

import pandas as pd
pd.set_option("display.max_colwidth", 70)
from fastapi.testclient import TestClient
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

from app.main import app
from app.service import CallRecord, RagService
from app.telemetry.monitor import Monitor
from app.telemetry.store import RequestLog
from app.telemetry.tracing import make_tracer
from app.settings import get_settings

s = get_settings()
print("model:", s.model, "| monitor db:", s.monitor_db.name, "| SLO:", s.slo_availability, "availability,",
      f"{s.slo_latency_target:.0%} of answers < {s.slo_latency_s}s")

model: llama3.1:8b | monitor db: monitor.db | SLO: 0.99 availability, 95% of answers < 10.0s


## 1. Observer Seam: Traffic → Request Log

Class 1-এর `RagService` প্রতিটা call শেষে একটা `CallRecord` বানায় আর প্রতিটা **observer**-কে দেয়। Monitoring পুরোটা একটা observer (`Monitor.observe`): quality signal হিসাব → SQLite row → Prometheus metric → trace। RAG core বা route-এ একটা লাইনও বদলায়নি।

এখানে দুটো observer: lifespan-এর আসল `Monitor` (SQLite, `/v1/stats`), আর একটা দ্বিতীয় `Monitor` যার trace একটা in-memory exporter-এ যায় (section 7)।

In [2]:
client = TestClient(app)
client.__enter__()                                    # lifespan: settings, monitor, warmup
svc = app.state.service
spans = InMemorySpanExporter()
tracing_monitor = Monitor(RequestLog(":memory:"), model=s.model, temperature=s.temperature,
                          num_predict=s.num_predict, tracer=make_tracer(None, exporter=spans))
svc.observers.append(tracing_monitor.observe)
print("observers:", [getattr(o, "__qualname__", o) for o in svc.observers])

pdf = (project_root / "samples" / "employee_handbook.pdf").read_bytes()
print(client.post("/v1/documents", files={"file": ("employee_handbook.pdf", pdf, "application/pdf")}).json())

/Users/salmansakib/Documents/Projects/interview-Prep-Case-Study/Job Ready AI Bootcamp/week 15/Class 2 Project/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1025 21:19:30.329000 76637 venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 10132.95it/s]

observers: ['Monitor.observe', 'Monitor.observe']
{'filename': 'employee_handbook.pdf', 'pages': 3, 'chunks': 7, 'new_chunks': 7, 'request_id': '3f79d68ff731'}


In [3]:
QUESTIONS = ["How many sick days do employees get?", "How many characters must a password have?",
             "How many days per week can I work remotely?", "How much is the home-office allowance?",
             "What is the meal limit when travelling?", "How long is parental leave?",
             "Can I carry over unused leave?", "Is MFA required for the VPN?",
             "What is the capital of France?", "Who won the 2018 World Cup?",
             "How many sick days do employees get?"]                 # last one repeats -> cache hit
for q in QUESTIONS:
    client.post("/v1/ask", json={"question": q})
with client.stream("POST", "/v1/ask/stream", json={"question": "How many weeks does the secondary caregiver get?"}) as r:
    "".join(r.iter_text())

log = app.state.monitor.log
df = pd.DataFrame(log.recent(50)).iloc[::-1].reset_index(drop=True)
df[["op", "outcome", "latency_s", "top_distance", "prompt_tokens", "completion_tokens", "citation_valid",
    "numbers_supported", "question"]]

,op,outcome,latency_s,top_distance,prompt_tokens,completion_tokens,citation_valid,numbers_supported,question
0,ask,answered,3.5565,12.53,394.0,16.0,1.0,1.0,How many sick days do employees get?
1,ask,answered,2.8844,19.46,367.0,15.0,1.0,1.0,How many characters must a password have?
2,ask,answered,3.1397,18.01,396.0,23.0,1.0,1.0,How many days per week can I work remotely?
3,ask,answered,3.5718,15.31,394.0,25.0,1.0,1.0,How much is the home-office allowance?
4,ask,answered,2.7346,24.45,394.0,17.0,1.0,1.0,What is the meal limit when travelling?
5,ask,answered,1.8379,22.26,378.0,24.0,1.0,1.0,How long is parental leave?
6,ask,answered,2.9908,25.20,379.0,21.0,1.0,1.0,Can I carry over unused leave?
7,ask,not_found,0.0388,27.06,NaN,NaN,NaN,NaN,Is MFA required for the VPN?
8,ask,not_found,0.0054,31.18,NaN,NaN,NaN,NaN,What is the capital of France?
9,ask,not_found,0.0043,27.18,NaN,NaN,NaN,NaN,Who won the 2018 World Cup?


**পড়ার নিয়ম:** প্রতিটা request-এর একটা row — Prometheus এটা পারে না (সে শুধু যোগফল/histogram রাখে)। "কোন প্রশ্নে কী হল" জানতে per-request log লাগে। `cached` row-এ token নেই, কারণ এবার LLM কাজই করেনি। Off-topic প্রশ্নে `not_found`, latency কয়েক ms।

## 2. Latency কোথায় যায়?

In [4]:
ans = df[df.outcome == "answered"]
stages = ans[["embed_ms", "search_ms", "prefill_ms", "decode_ms"]].median().round(1)
print("median ms per stage:"); print(stages.to_string())
print(f"\nLLM share of answered latency: {ans.llm_ms.sum() / (ans.latency_s.sum() * 1000):.1%}")
ans[["question", "prompt_tokens", "prefill_ms", "decode_ms", "tokens_per_s", "ttft_ms"]]

median ms per stage:
embed_ms        58.4
search_ms        1.3
prefill_ms    1863.9
decode_ms      980.6

LLM share of answered latency: 97.1%


,question,prompt_tokens,prefill_ms,decode_ms,tokens_per_s,ttft_ms
0,How many sick days do employees get?,394.0,2539.9,962.6,16.6,NaN
1,How many characters must a password have?,367.0,1755.5,812.4,18.5,NaN
2,How many days per week can I work remotely?,396.0,1938.0,1094.5,21.0,NaN
3,How much is the home-office allowance?,394.0,2174.6,1276.0,19.6,NaN
4,What is the meal limit when travelling?,394.0,1789.8,868.9,19.6,NaN
5,How long is parental leave?,378.0,435.2,1257.3,19.1,NaN
6,Can I carry over unused leave?,379.0,1940.3,998.5,21.0,NaN
11,How many weeks does the secondary caregiver get?,381.0,142.7,699.0,21.5,162.3


**দুটো জিনিস লক্ষ্য করুন:**

* Embed + search মিলে কয়েক ms; প্রায় পুরো সময় LLM-এর। Monitoring-এ তাই stage আলাদা করে মাপা জরুরি — শুধু মোট latency দেখলে বোঝা যায় না কোথায় হাত দিতে হবে।
* `prefill_ms` দুই রকম: কিছু কয়েকশো-হাজার ms, কিছু ~৫০ ms। Ollama আগের prompt-এর KV cache রেখে দেয় — prefix মিলে গেলে prefill প্রায় বিনামূল্যে (Class 1 notebook §5)। তাই TTFT-এর বণ্টন **bimodal**: গড় দেখলে দুটোর কোনোটাই বোঝা যায় না, percentile দেখুন।

## 3. Prometheus Metrics

`/metrics` হলো text format; Prometheus প্রতি ১৫ s-এ এটা "scrape" করে, তারপর PromQL দিয়ে rate, percentile, alert। নিচে নিজেরাই scrape করে parse করা হলো।

In [5]:
from prometheus_client.parser import text_string_to_metric_families

text = client.get("/metrics").text
fams = {f.name: f for f in text_string_to_metric_families(text)}
print(len(fams), "metric families, e.g.:", sorted(n for n in fams if n.startswith("rag_"))[:8])
pd.DataFrame([{"labels": s_.labels, "value": s_.value} for s_ in fams["rag_http_requests"].samples
              if s_.name.endswith("_total")])

23 metric families, e.g.: ['rag_answers', 'rag_answers_created', 'rag_decode_tokens_per_second', 'rag_decode_tokens_per_second_created', 'rag_feedback', 'rag_http_requests', 'rag_http_requests_created', 'rag_lexical_support_ratio']


,labels,value
0,"{'method': 'POST', 'route': '/v1/documents', 'status': '201'}",1.0
1,"{'method': 'POST', 'route': '/v1/ask', 'status': '200'}",11.0
2,"{'method': 'POST', 'route': '/v1/ask/stream', 'status': '200'}",1.0


In [6]:
llm = [x for x in fams["rag_stage_duration_seconds"].samples
       if x.name.endswith("_bucket") and x.labels["stage"] == "llm"]
buckets = pd.DataFrame([{"le": x.labels["le"], "cumulative_count": x.value} for x in llm])
buckets.T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
le,0.001,0.0025,0.005,0.01,0.025,0.05,0.1,0.25,0.5,1.0,1.0,2.0,4.0,8.0,16.0,32.0,64.0,120.0,+Inf
cumulative_count,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,4.0,16.0,16.0,16.0,16.0,16.0,16.0,16.0


**তিনটা নিয়ম, তিনটা ফাঁদ:**

* **Bucket:** `prometheus_client`-এর default bucket শেষ হয় **10 s**-এ। Local 8B model-এ লম্বা উত্তর বা queue-তে অপেক্ষা সহজেই 10 s পার হয় — তখন সব `+Inf`-এ পড়ে, আর `histogram_quantile` বলে p95 = 10 s (মিথ্যা)। এখানে 64 s আর 120 s পর্যন্ত bucket।
* **Cardinality:** label-এ route **template** (`/v1/ask`), raw path না। নিচে একটা অজানা path পাঠিয়ে দেখুন — label হয় `unmatched`, নতুন time series না। User id, প্রশ্ন, request id কখনো label না।
* **Unit:** seconds আর `_total` — PromQL-এর `rate()` আর dashboard সেটাই ধরে নেয়।

In [7]:
for path in ["/v1/documents/abc", "/v1/documents/xyz", "/some/random/path"]:
    client.get(path)
lines = client.get("/metrics").text.splitlines()
routes = sorted({l.split('route="')[1].split('"')[0] for l in lines if l.startswith("rag_http_requests_total{")})
print("distinct route labels after 3 random paths:", routes)

# default buckets vs ours: where would a 14 s answer land?
from prometheus_client import Histogram
print("default buckets:", Histogram.DEFAULT_BUCKETS)
print("a 14 s answer -> default bucket:", next(b for b in Histogram.DEFAULT_BUCKETS if 14 <= b),
      "| ours:", next(b for b in (0.1, 0.25, 0.5, 1, 2, 4, 8, 16, 32, 64, 120) if 14 <= b))

distinct route labels after 3 random paths: ['/metrics', '/v1/ask', '/v1/ask/stream', '/v1/documents', 'unmatched']
default buckets: (0.005, 0.01, 0.025, 0.05, 0.075, 0.1, 0.25, 0.5, 0.75, 1.0, 2.5, 5.0, 7.5, 10.0, inf)
a 14 s answer -> default bucket: inf | ours: 16


## 4. Cheap Quality Signals

HTTP 200 মানেই ভালো উত্তর না। প্রতিটা request-এ (microsecond-এ, LLM ছাড়া) তিনটা চুপচাপ ব্যর্থতা ধরা হয়:

| Signal | কী ধরে | কী ধরে না |
|---|---|---|
| `citation_valid` | `[7]` অথচ passage মাত্র ৪টা; কোনো citation নেই | ভুল কিন্তু cite করা দাবি |
| `numbers_supported` | passage-এ নেই এমন সংখ্যা (দিন, USD, %) | কথায় লেখা সংখ্যা ("twelve") |
| `lexical_support` | passage-এ নেই এমন শব্দ | paraphrase-এ কম দেখায় |
| `truncated` | `num_predict`-এ কেটে যাওয়া উত্তর | — |

নিচে আসল hits নিয়ে ইচ্ছে করে ভুল উত্তর বসানো হলো:

In [8]:
from app.rag.engine import AskResult
from app.telemetry.quality import assess

engine = svc.backend.engine
hits, _ = engine.retrieve("How many sick days do employees get?")
PLANTED = {
    "correct": "Employees get 12 paid sick days per year [1].",
    "paraphrase": "You get twelve paid sick days each year [1].",
    "invented number 18": "Employees get 18 paid sick days per year [1].",
    "invented number 15": "Employees get 15 paid sick days per year [1].",
    "bad citation": "Employees get 12 paid sick days per year [9].",
    "no citation": "Employees get 12 paid sick days per year.",
    "invented claim": "Employees get 12 paid sick days per year, and unused sick days can be cashed out [1].",
}
rows = []
for name, text in PLANTED.items():
    sig = assess(AskResult(text, found=True, hits=hits), cached=False, ok=True, num_predict=s.num_predict)
    rows.append({"case": name, "citation_valid": sig.citation_valid, "numbers_supported": sig.numbers_supported,
                 "unsupported_numbers": sig.unsupported_numbers, "lexical_support": sig.lexical_support})
pd.DataFrame(rows)

,case,citation_valid,numbers_supported,unsupported_numbers,lexical_support
0,correct,True,True,,1.000
1,paraphrase,True,True,,0.800
2,invented number 18,True,False,18,1.000
3,invented number 15,True,True,,1.000
4,bad citation,False,True,,1.000
5,no citation,False,True,,1.000
6,invented claim,True,True,,0.857


**পড়ার নিয়ম:** "18" আর দুটো citation সমস্যা সস্তা check-ই ধরে। কিন্তু দুটো ভুল পার পেয়ে যায়:

* **"15"** — কারণ অন্য একটা passage-এ "15:00" (remote work-এর core hours) আছে। Check শুধু প্রমাণ করে সংখ্যাটা context-এ *কোথাও* আছে, সঠিক তথ্যের সাথে আছে কিনা না।
* **"invented claim"** (cash out) — সংখ্যা ঠিক, citation ঠিক, শব্দগুলো বেশিরভাগ passage-এ আছে।

সস্তা check দরকারি (প্রতিটা request, শূন্য খরচ), কিন্তু যথেষ্ট না। এই ফাঁক LLM judge (section 6) পূরণ করে।

## 5. Not-Found Review: Guardrail-এর False Negative

Guardrail (`max_distance`) off-topic প্রশ্নে LLM ডাকা বন্ধ করে। কিন্তু উল্টো ভুলও হয়: handbook-এ উত্তর আছে, তবু "not found"। এটা error rate-এ দেখা যায় না (HTTP 200) — not-found row-গুলো চোখে দেখতে হয়।

In [9]:
nf = pd.DataFrame(log.rows(0))
nf = nf[nf.outcome == "not_found"][["question", "top_distance"]]
nf["threshold"] = s.max_distance
nf["margin"] = (nf.top_distance - s.max_distance).round(2)
nf

,question,top_distance,threshold,margin
7,Is MFA required for the VPN?,27.06,26.0,1.06
8,What is the capital of France?,31.18,26.0,5.18
9,Who won the 2018 World Cup?,27.18,26.0,1.18


VPN/MFA প্রশ্নের উত্তর handbook-এ আছে ("Multi-factor authentication is mandatory for email, VPN…"), কিন্তু distance threshold-এর উপরে → false negative। কেন? নিচে একই প্রশ্ন দুইভাবে:

In [10]:
engine = svc.backend.engine
for q in ["Is MFA required for the VPN?", "Is multi-factor authentication required for the VPN?"]:
    h, _ = engine.retrieve(q)
    print(f"{h[0].distance:6.2f}  {'not found' if h[0].distance > s.max_distance else 'answered':9}  {q}")

 27.06  not found  Is MFA required for the VPN?
 17.77  answered   Is multi-factor authentication required for the VPN?


ছোট embedding model (`paraphrase-MiniLM-L3-v2`) "MFA" acronym-কে "multi-factor authentication"-এর সাথে মেলাতে পারে না। Fix-এর পথ: threshold, ভালো embedding model, hybrid search (BM25 acronym-এ exact match ধরে), বা acronym expansion। Threshold বাড়ালে এটা ঢুকবে, কিন্তু off-topic প্রশ্নও ঢুকতে পারে — Class 1-এর calibration table আবার দেখুন। **Monitoring-এর কাজ এখানে সিদ্ধান্ত নেওয়া না, সিদ্ধান্তের data দেওয়া:** not-found rate বাড়লে (`RagNotFoundRateHigh` ticket) এই table-টাই খুলতে হয়।

## 6. Offline LLM Judge

Judge = আরেকটা LLM call যেটা জিজ্ঞেস করে "উত্তরের প্রতিটা দাবি কি passage-এ আছে?"। এটা **request-এর ভেতরে না**: Ollama একসাথে একটা request চালায়, judge ভেতরে থাকলে user-এর latency দ্বিগুণ। তাই log থেকে পরে, batch-এ (`python -m app.telemetry.judge`)।

Project বানানোর সময় তিনটা design মাপা হয়েছিল (llama3.1:8b, ৮টা আসল উত্তর + ৩টা planted):

| Design | ফল |
|---|---|
| শুধু `{"supported": bool}` | একবার `true` বলে একই উত্তরে unsupported claim লিখেছিল — নিজের সাথে অসঙ্গত |
| আগে "unsupported claim-এর তালিকা", তারপর verdict | সব claim-ই তালিকায় দিল → ১১-র ১১টাই fail |
| **claim ধরে ধরে: claim → passage থেকে হুবহু evidence → verdict** + code-এ quote check | **১১/১১ ঠিক** |

নিচে চূড়ান্ত design আসল model-এ:

In [11]:
import ollama
from app.telemetry import judge

for name in ["correct", "paraphrase", "invented number 15", "invented claim"]:     # planted rows -> same log
    res = AskResult(PLANTED[name], found=True, hits=hits, timings_ms={"embed_ms": 1, "search_ms": 1, "llm_ms": 1})
    app.state.monitor.observe(CallRecord("ask", f"planted:{name}", 0.0, True, None, res,
                                         question="How many sick days do employees get?"))

t0 = time.perf_counter()
verdicts = judge.run(log, engine.store, ollama.Client(host=s.ollama_host), s.model, limit=30)
print(f"judged {len(verdicts)} answers in {time.perf_counter() - t0:.0f}s")
jdf = pd.DataFrame(log.rows(0))
jdf = jdf[jdf.judge_supported.notna()][["request_id", "answer", "numbers_supported", "judge_supported", "judge_reason"]]
jdf.assign(answer=jdf.answer.str[:60])

judged 12 answers in 64s


,request_id,answer,numbers_supported,judge_supported,judge_reason
0,2c9a7829e5cc,"According to [1], employees get 12 paid sick days per year.",1.0,1.0,
1,adc00bd89a65,"According to [1], passwords must have at least 14 characters",1.0,1.0,
2,9412932c65fe,"According to [1], employees may work remotely up to 3 days p",1.0,1.0,
3,6fa48c47dd89,"The home-office allowance is 300 USD, which covers a desk, c",1.0,1.0,
4,76f11186a131,Meals are reimbursed up to 50 USD per day. [1],1.0,1.0,
5,1c4048e7b5d4,"The primary caregiver receives 16 weeks of paid leave, and t",1.0,1.0,
6,5ac76aeca2f6,"According to [1], up to 5 unused days can be carried over to",1.0,1.0,
11,ab1f64eda5fd,The secondary caregiver receives 4 weeks of paid leave. [1],1.0,1.0,
12,planted:correct,Employees get 12 paid sick days per year [1].,1.0,1.0,
13,planted:paraphrase,You get twelve paid sick days each year [1].,1.0,1.0,


**পড়ার নিয়ম:** "invented number 15" আর "invented claim" — দুটোই সস্তা check পাস করেছিল — judge ধরে, কারণ সে প্রতিটা claim-কে নির্দিষ্ট passage-বাক্যের সাথে মেলায়। "paraphrase" (twelve) judge পাস করায়, যদিও `numbers_supported` সেটা দেখতে পায় না (সংখ্যা নেই)।

⚠️ Judge আর উত্তরদাতা একই model → **self-preference bias**। Judge-এর score absolute সত্য না — trend দেখুন, আর মাঝে মাঝে ২০-৩০টা উত্তর নিজে label করে judge-এর সাথে মিলিয়ে নিন।

## 7. Traces: Span Tree + GenAI Attributes

একটা trace = একটা request; ভেতরে span = প্রতিটা stage। Service layer প্রতিটা stage-এর আসল সময় মেপে রাখে, তাই request শেষে সেই মাপ থেকে explicit start/end time দিয়ে span বানানো হয় — RAG core-এ tracing code ঢোকাতে হয় না। Production-এ একই span OTLP দিয়ে Phoenix বা Langfuse-এ যায় (`RAG_OTLP_ENDPOINT`)।

In [12]:
finished = spans.get_finished_spans()
roots = [x for x in finished if x.parent is None and x.name in ("rag.ask", "rag.stream")]
answered_root = next(r for r in roots if r.attributes.get("rag.outcome") == "answered")

def show(span, depth=0):
    ms = (span.end_time - span.start_time) / 1e6
    print(f"{'  ' * depth}{span.name:22} {ms:8.1f} ms   {span.attributes.get('openinference.span.kind', '')}")
    for child in sorted((x for x in finished if x.parent and x.parent.span_id == span.context.span_id),
                        key=lambda x: x.start_time):
        show(child, depth + 1)

print(len(finished), "spans in", len(roots), "request traces\n")
show(answered_root)
chat = next(x for x in finished if x.parent and x.parent.span_id == answered_root.context.span_id and x.name.startswith("chat"))
{k: v for k, v in chat.attributes.items() if k.startswith("gen_ai.")}

46 spans in 12 request traces

rag.ask                  3556.5 ms   CHAIN
  embed                       7.8 ms   EMBEDDING
  retrieve                    1.6 ms   RETRIEVER
  chat llama3.1:8b         3546.6 ms   LLM


{'gen_ai.operation.name': 'chat',
 'gen_ai.provider.name': 'ollama',
 'gen_ai.request.model': 'llama3.1:8b',
 'gen_ai.request.temperature': 0.2,
 'gen_ai.request.max_tokens': 512,
 'gen_ai.usage.input_tokens': 394,
 'gen_ai.usage.output_tokens': 16}

`gen_ai.*` নামগুলো OpenTelemetry-র GenAI semantic conventions থেকে — status এখনো **Development**, অর্থাৎ নাম বদলাতে পারে (যেমন `gen_ai.system` → `gen_ai.provider.name`)। তাই project-এ সব নাম একটাই file-এ (`tracing.py`) — বদলালে এক জায়গায় ঠিক করতে হয়।

## 8. SLO + Burn Rate: কখন Page করবেন?

**SLO** = লক্ষ্য, যেমন "৯৯% request সফল"। বাকি ১% = **error budget**। **Burn rate** = error ratio ÷ budget: 1× মানে budget ঠিক মাস শেষে শেষ হবে; 14.4× মানে ২ দিনে। Google SRE Workbook-এর multiwindow নিয়ম: **1h আর 5m দুটোতেই** burn > 14.4 হলে page — লম্বা window নিশ্চিত করে এটা সত্যিকারের সমস্যা, ছোট window নিশ্চিত করে সমস্যা *এখনো* চলছে (ঠিক হয়ে গেলে alert দ্রুত থামে)।

নিচে FakeBackend (প্রতি উত্তর ০.৩ s, ১টা slot, ০.০৫ s queue timeout) দিয়ে একটা overload burst — Ollama লাগে না:

In [13]:
from app.backend import FakeBackend
from app.errors import Overloaded

slo_log = RequestLog(":memory:")
slo_mon = Monitor(slo_log, model="fake", temperature=0, num_predict=512)
fake = RagService(FakeBackend(delay=0.3), max_concurrent=1, queue_timeout_s=0.05, cache_size=0,
                  observers=[slo_mon.observe])
fake.backend.docs["hb.pdf"] = "x"

async def one(i):
    try:
        await fake.ask(f"q{i}")
    except Overloaded:
        pass

async def traffic():
    for i in range(10):                      # healthy: one at a time
        await one(i)
    await asyncio.gather(*(one(100 + i) for i in range(8)))      # burst: 8 at once, 1 slot

await traffic()
summary = slo_log.summary(3600, slo_availability=0.99)
print("outcomes:", summary["outcomes"])
pd.DataFrame({k: v["burn_rate"] | {"alerts": v["alerts"]} for k, v in summary["slo"].items()}).T

outcomes: {'answered': 11, 'not_found': 0, 'cached': 0, 'degraded': 0, 'error': 7, 'disconnected': 0}


,5m,30m,1h,6h,alerts
availability,38.89,38.89,38.89,38.89,"[page_fast, page_slow]"
latency,0.0,0.0,0.0,0.0,[]


**পড়ার নিয়ম:** ১৮টার মধ্যে ৭টা 503 → error ratio ≈ 0.39 → burn ≈ 39× (budget 1%) → `page_fast` আর `page_slow` দুটোই। Production-এ একই হিসাব Prometheus করে `monitoring/alerts.yml`-এর rule দিয়ে; `promtool test rules monitoring/alerts_test.yml` সেই rule-এর unit test (20% error → page, 1% → চুপ)।

Quality signal (judge, not-found rate) কখনো page করে না — sample ছোট, noise বেশি → শুধু ticket।

In [14]:
client.__exit__(None, None, None)
print("closed; scratch data in", tmp)

closed; scratch data in /var/folders/df/xvbb4c0j2zn25ysqm7jn_lw40000gn/T/rag_mon_nb_n6t4vvp4


## 📝 Key Takeaways

* **Seam আগে, tool পরে।** Service layer-এ একটা observer hook থাকলে monitoring যোগ করতে RAG code ছুঁতে হয় না — SQLite, Prometheus, OTel সব একই observer থেকে।
* **Stage ধরে মাপুন।** মোট latency কিছু বলে না; LLM ~৯৯%, আর prefill cache-এর কারণে TTFT bimodal।
* **Prometheus = aggregate, log = প্রতিটা request।** Alert-এর জন্য প্রথমটা, "কোন উত্তর ভুল" খুঁজতে দ্বিতীয়টা। Bucket LLM-আকারের, label সীমিত।
* **Quality তিন স্তরে।** সস্তা check সব request-এ; LLM judge offline আর sample করে; মানুষ মাঝে মাঝে judge-কে যাচাই করে।
* **Judge design-ও মাপতে হয়।** একই model, তিনটা prompt, ফল ০/১১ থেকে ১১/১১।
* **Page শুধু user-এর ব্যথায়** (availability, latency SLO burn); quality signal → ticket।

## 🧠 Brain Teasers

1. Section 5-এর MFA প্রশ্ন ঢোকাতে `max_distance` কত করতে হবে? সেই মানে Class 1-এর off-topic প্রশ্নগুলো কী হয়?
2. `rag_http_requests_total`-এ `question` label যোগ করলে ১০,০০০ জন user-এর পর কয়টা time series হবে? কেন সেটা সমস্যা?
3. Judge-এর `Claim` model-এ `supported` field-কে `evidence`-এর আগে আনুন। Section 6 আবার চালান — ফল বদলায়?
4. Latency SLO "৯৫% উত্তর < 10 s"। Section 8-এ `FakeBackend(delay=12)` দিয়ে চালালে কোন alert আসে, আর কেন availability alert আসে না?